In [7]:
# --- Imports & config path ---
import os, sys, re, csv
from pathlib import Path
from urllib.parse import urlparse, urljoin, unquote
from bs4 import BeautifulSoup
from bs4.element import NavigableString, Tag

project_root = Path.cwd().parent.parent   # adjust ../.. as needed
sys.path.append(str(project_root))

print("Project root added to sys.path:", project_root)
try:
    import config  # expects BASE_URL, FANDOM_DATA_DIR, LINKS_FILE
except Exception as e:
    raise RuntimeError("config.py not found or invalid") from e

# --- Paths ---
BASE_URL = config.BASE_URL.rstrip("/")
domain_full = urlparse(BASE_URL).netloc
domain = domain_full.split(".")[0]
FANDOM_DATA_DIR = Path(config.FANDOM_DATA_DIR)

LINKS_FILE = Path(config.LINKS_FILE)
if not LINKS_FILE.is_absolute():
    LINKS_FILE = FANDOM_DATA_DIR / LINKS_FILE.name

HTML_DIR  = FANDOM_DATA_DIR / f"{domain}_fandom_html"
SPANS_DIR = FANDOM_DATA_DIR / f"{domain}_fandom_spans"
SPANS_DIR.mkdir(parents=True, exist_ok=True)

MASTER_CSV = FANDOM_DATA_DIR / f"master_spans_{domain}.csv"

FIELDNAMES = [
    "article_id","title","paragraph_id","paragraph_text","anchor_ix",
    "link_text","start","end","link_type","resolved_url","page_url",
    "cleaned_url","article_id_of_internal_link"
]

print("BASE_URL:", BASE_URL)
print("HTML_DIR:", HTML_DIR.exists(), HTML_DIR)
print("SPANS_DIR:", SPANS_DIR.exists(), SPANS_DIR)
print("LINKS_FILE:", LINKS_FILE.exists(), LINKS_FILE)

Project root added to sys.path: /home
BASE_URL: https://alldimensions.fandom.com/wiki/All_dimensions_Wiki
HTML_DIR: True /home/sundeep/Fandom-Span-Identification-and-Retrieval/1.Fandom_Dataset_Collection/raw_data/alldimensions_fandom_data/alldimensions_fandom_html
SPANS_DIR: True /home/sundeep/Fandom-Span-Identification-and-Retrieval/1.Fandom_Dataset_Collection/raw_data/alldimensions_fandom_data/alldimensions_fandom_spans
LINKS_FILE: True /home/sundeep/Fandom-Span-Identification-and-Retrieval/1.Fandom_Dataset_Collection/raw_data/alldimensions_fandom_data/alldimensions_articles_list.txt


In [8]:
# Quick path checks
assert HTML_DIR.exists() and HTML_DIR.is_dir(), f"Missing HTML_DIR: {HTML_DIR}"
assert SPANS_DIR.exists() and SPANS_DIR.is_dir(), f"Missing SPANS_DIR: {SPANS_DIR}"
print("✓ Paths OK")

# Peek a few html files
html_files = sorted([p for p in HTML_DIR.glob("*.html") if p.is_file()])
print(f"HTML count: {len(html_files)}")
assert len(html_files) > 0, "No .html files found"
for p in html_files[:3]:
    print(" -", p.name)

✓ Paths OK
HTML count: 3714
 - -.html
 - -Flaws_in_the_World-.html
 - -One_Who_Stands_Before_God-.html


In [1]:
def extract_article_id(raw_html: str):
    soup = BeautifulSoup(raw_html, "html.parser")
    # 1. Direct meta attributes
    tag = soup.find(attrs={"wgArticleId": True}) or soup.find(attrs={"wgArticleID": True})
    if tag:
        val = tag.get("wgArticleId") or tag.get("wgArticleID")
        if val and str(val).isdigit():
            return int(val)
    # 2. Look inside <script> blocks
    for sc in soup.find_all("script"):
        txt = sc.string or sc.get_text() or ""
        m = re.search(r'"wgArticleId"\s*:\s*(\d+)', txt)
        if m:
            return int(m.group(1))
    return None

In [9]:
def extract_article_id(raw_html: str):
    soup = BeautifulSoup(raw_html, "html.parser")
    for sc in soup.find_all("script"):
        txt = sc.string or sc.get_text() or ""
        m = re.search(r'"wgArticleId"\s*:\s*(\d+)', txt)
        if m:
            return int(m.group(1))
    return None

def extract_title(soup: BeautifulSoup) -> str:
    h1 = soup.select_one("#firstHeading") or soup.find("h1")
    if h1:
        return h1.get_text(strip=True)
    if soup.title:
        return soup.title.get_text(strip=True)
    return ""

def make_page_url(html_path: Path) -> str:
    slug = html_path.stem
    return urljoin(BASE_URL + "/", f"wiki/{slug}")

def classify_link(href: str, page_url: str) -> str:
    if not href:
        return "unknown"
    if href.startswith("#"):
        return "anchor"
    if href.startswith("/wiki/") or href.startswith(BASE_URL + "/wiki/"):
        abs_url = urljoin(BASE_URL + "/", href)
        norm_abs  = unquote(abs_url).rstrip("/").replace("_", " ")
        norm_page = unquote(page_url).rstrip("/").replace("_", " ")
        return "self" if norm_abs == norm_page else "internal"
    if href.startswith(("http://", "https://")):
        return "external"
    return "unknown"

def load_aid_map(path: Path) -> dict:
    if not path.exists():
        return {}
    aid_map = {}
    with path.open("r", encoding="utf-8", newline="") as f:
        reader = csv.DictReader(f)
        cols = {c.lower(): c for c in (reader.fieldnames or [])}
        aid_col = cols.get("article_id") or cols.get("aid") or cols.get("id")
        slug_like = cols.get("slug") or cols.get("title") or cols.get("page_url") or cols.get("cleaned_url")
        if not aid_col or not slug_like:
            return {}
        for row in reader:
            try:
                aid = str(int(row[aid_col]))
            except Exception:
                continue
            key = (row.get(slug_like) or "").strip()
            if not key:
                continue
            key_u = unquote(key)
            for k in {key, key_u, key_u.replace(" ", "_"), key_u.replace("_", " ")}:
                aid_map[k] = aid
            m = re.search(r"/wiki/([^?#/]+.*)$", key_u)
            if m:
                s = m.group(1)
                for k in {s, s.replace(" ", "_"), s.replace("_", " ")}:
                    aid_map[k] = aid
    return aid_map

In [10]:
def extract_paragraph_spans(
    soup: BeautifulSoup,
    page_url: str,
    article_id: int | None,
    title: str | None,
    aid_map: dict | None = None,
):
    rows = []
    for pid, block in enumerate(soup.select("p"), start=1):
        para = ""
        spans = []
        anchor_ix = 0

        for node in block.descendants:
            if isinstance(node, NavigableString):
                if node.parent and node.parent.name != "a":
                    para += str(node)
            elif isinstance(node, Tag) and node.name == "a":
                raw = node.get_text()
                if not raw:
                    continue
                start = len(para)
                para += raw
                end = start + len(raw)

                href = node.get("href", "") or ""
                resolved_url = urljoin(BASE_URL + "/", href) if href else ""
                cleaned_url = unquote(resolved_url.split("#", 1)[0].rstrip("/")) if resolved_url else ""
                link_type = classify_link(href, page_url)

                target_aid = ""
                if link_type in ("internal", "self") and cleaned_url:
                    m = re.search(r"/wiki/([^?#/]+.*)$", cleaned_url)
                    if m and aid_map:
                        slug = m.group(1)
                        target_aid = (
                            aid_map.get(slug)
                            or aid_map.get(slug.replace("_", " "))
                            or aid_map.get(slug.replace(" ", "_"))
                            or ""
                        )

                spans.append(
                    {
                        "article_id": article_id,
                        "title": title,
                        "paragraph_id": pid,
                        "paragraph_text": None,  # fill after paragraph is complete
                        "anchor_ix": anchor_ix,
                        "link_text": raw,
                        "start": start,
                        "end": end,
                        "link_type": link_type,
                        "resolved_url": resolved_url,
                        "page_url": page_url,
                        "cleaned_url": cleaned_url,
                        "article_id_of_internal_link": target_aid,
                    }
                )
                anchor_ix += 1

        para_text = para
        for s in spans:
            if 0 <= s["start"] < s["end"] <= len(para_text):
                s["paragraph_text"] = para_text
                rows.append(s)
    return rows

In [11]:
aid_map = load_aid_map(LINKS_FILE)

html_files = sorted([p for p in HTML_DIR.glob("*.html") if p.is_file()])
print("HTML files:", len(html_files))

with MASTER_CSV.open("w", encoding="utf-8", newline="") as master_fh:
    master_writer = csv.DictWriter(master_fh, fieldnames=FIELDNAMES, extrasaction="ignore")
    master_writer.writeheader()

    for i, html_path in enumerate(html_files, start=1):
        try:
            raw = html_path.read_text(encoding="utf-8", errors="ignore")
            article_id = extract_article_id(raw)
            soup = BeautifulSoup(raw, "html.parser")
            title = extract_title(soup)
            page_url = make_page_url(html_path)

            rows = extract_paragraph_spans(
                soup=soup,
                page_url=page_url,
                article_id=article_id,
                title=title,
                aid_map=aid_map,
            )

            out_csv = SPANS_DIR / f"{html_path.stem}_spans.csv"
            with out_csv.open("w", encoding="utf-8", newline="") as f:
                w = csv.DictWriter(f, fieldnames=FIELDNAMES, extrasaction="ignore")
                w.writeheader()
                w.writerows(rows)

            for r in rows:
                master_writer.writerow(r)

            if i % 50 == 0:
                print(f"[{i}/{len(html_files)}] processed")
        except Exception as e:
            print(f"Error on {html_path.name}: {e}")

print("Master CSV:", MASTER_CSV)



HTML files: 3714
[50/3714] processed
[100/3714] processed
[150/3714] processed
[200/3714] processed
[250/3714] processed
[300/3714] processed
[350/3714] processed
[400/3714] processed
[450/3714] processed
[500/3714] processed
[550/3714] processed
[600/3714] processed
[650/3714] processed
[700/3714] processed
[750/3714] processed
[800/3714] processed
[850/3714] processed
[900/3714] processed
[950/3714] processed
[1000/3714] processed
[1050/3714] processed
[1100/3714] processed
[1150/3714] processed
[1200/3714] processed
[1250/3714] processed
[1300/3714] processed
[1350/3714] processed
[1400/3714] processed
[1450/3714] processed
[1500/3714] processed
[1550/3714] processed
[1600/3714] processed
[1650/3714] processed
[1700/3714] processed
[1750/3714] processed
[1800/3714] processed
[1850/3714] processed
[1900/3714] processed
[1950/3714] processed
[2000/3714] processed
[2050/3714] processed
[2100/3714] processed
[2150/3714] processed
[2200/3714] processed
[2250/3714] processed
[2300/3714] p

In [20]:
from collections import Counter

REQ_FIELDS = {
    "article_id","title","paragraph_id","paragraph_text","anchor_ix",
    "link_text","start","end","link_type","resolved_url","page_url",
    "cleaned_url","article_id_of_internal_link"
}

def check_rows_basic(rows, page_url:str):
    # schema
    for i, r in enumerate(rows[:5]):
        missing = REQ_FIELDS - set(r.keys())
        assert not missing, f"Row {i} missing fields: {missing}"
    # counts
    c = Counter(r["link_type"] for r in rows)
    print("link_type split:", dict(c))
    # bounds & slice-matches
    bad_bounds = 0; mismatch = 0
    for r in rows:
        s,e = r["start"], r["end"]
        t = r["paragraph_text"] or ""
        if not (isinstance(s,int) and isinstance(e,int)):
            bad_bounds += 1; continue
        if not (0 <= s < e <= len(t)):
            bad_bounds += 1; continue
        if t[s:e] != r["link_text"]:
            mismatch += 1
    print(f"bounds issues: {bad_bounds} | text mismatches: {mismatch}")
    # URL hygiene
    has_abs = sum(bool(r["resolved_url"]) for r in rows)
    has_clean = sum(bool(r["cleaned_url"]) for r in rows)
    print(f"resolved_url set: {has_abs}/{len(rows)} | cleaned_url set: {has_clean}/{len(rows)}")
    # anchor indexing monotonic
    by_para = {}
    for r in rows:
        key = (r["paragraph_id"])
        by_para.setdefault(key, []).append(r["anchor_ix"])
    non_mono = sum(sorted(ixs)!=ixs for ixs in by_para.values())
    print(f"non-monotonic anchor_ix paragraphs: {non_mono}")

def check_internal_targets(rows):
    internals = [r for r in rows if r["link_type"] in ("internal","self")]
    missing = sum(1 for r in internals if not r["article_id_of_internal_link"])
    print(f"internal/self: {len(internals)} | missing target AID: {missing}")

In [21]:
import pandas as pd

df = pd.read_csv(MASTER_CSV)
print("master rows:", len(df))
# required columns present
missing_cols = [c for c in FIELDNAMES if c not in df.columns]
assert not missing_cols, f"Missing columns in master: {missing_cols}"
print("✓ schema OK")

# types & bounds
for c in ["start","end","paragraph_id","anchor_ix"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")
bad_na = df[["start","end","paragraph_id","anchor_ix"]].isna().sum().sum()
print("numeric NA count:", int(bad_na))

neg_start = (df["start"] < 0).sum()
rev = (df["end"] <= df["start"]).sum()
print("neg starts:", int(neg_start), "| non-positive lengths:", int(rev))

# slice-match sample (first 200)
m = 0
for _, r in df.head(200).iterrows():
    t = str(r["paragraph_text"])
    s,e = int(r["start"]), int(r["end"])
    if not (0 <= s < e <= len(t)):
        m += 1
    elif t[s:e] != str(r["link_text"]):
        m += 1
print("sample text mismatches (200):", m)

# link_type distribution
print(df["link_type"].value_counts(dropna=False).to_dict())

# duplicates: same paragraph & span & URL
dupe_cols = ["article_id","paragraph_id","start","end","cleaned_url"]
dupes = df.duplicated(subset=dupe_cols).sum()
print("exact dupes (article_id,paragraph_id,start,end,cleaned_url):", int(dupes))

# internal targets missing
mask_int = df["link_type"].isin(["internal","self"])
missing_target = (df.loc[mask_int, "article_id_of_internal_link"].astype(str).str.len()==0).sum()
print("internal/self rows:", int(mask_int.sum()), "| missing AID targets:", int(missing_target))

master rows: 30333
✓ schema OK
numeric NA count: 0
neg starts: 0 | non-positive lengths: 0
sample text mismatches (200): 0
{'internal': 25721, 'external': 3570, 'anchor': 1041, 'unknown': 1}
exact dupes (article_id,paragraph_id,start,end,cleaned_url): 7
internal/self rows: 25721 | missing AID targets: 0
